In [1]:
!pip install tensorflow
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.layers import Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras import backend as K
from tensorflow import keras
K.set_image_data_format('channels_first')


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
inc_model = InceptionV3(include_top=False ,weights='imagenet', input_shape=(3, 290, 290))
for layer in inc_model.layers[:205]:
    layer.trainable = False

In [3]:
train_generator = keras.utils.image_dataset_from_directory(r'C:\Users\user\Desktop\DatasetCards\Training',
                                                           image_size=(290, 290),
                                                           batch_size=16,
                                                           class_names=None,
                                                           shuffle=True)

test_generator = keras.utils.image_dataset_from_directory(r'C:\Users\user\Desktop\DatasetCards\Test',
                                                                image_size=(290, 290),
                                                                batch_size=16,
                                                                class_names=None,
                                                                shuffle=True)

Found 7484 files belonging to 2 classes.
Found 100 files belonging to 2 classes.


In [4]:
inputs = keras.Input(shape=(3, 290, 290))
x = inc_model(inputs, training=False)
x = Dense(512, activation='relu')(x)
x = Dense(64, activation='relu', name='dense_one')(x)
x = Dropout(0.5, name='dropout_one')(x)
x = Dense(64, activation='relu', name='dense_two')(x)
x = Dropout(0.5, name='dropout_two')(x)
x = GlobalAveragePooling2D()(x)
predictions = Dense(1, activation='sigmoid')(x)

model = Model(inputs=inputs, outputs=predictions)

model.compile(loss='binary_crossentropy',
              optimizer=SGD(learning_rate=1e-4, momentum=0.9),
              metrics=['accuracy'])

In [5]:
model.fit(train_generator, batch_size=32, epochs=5, verbose=1)

score = model.evaluate(test_generator, batch_size=32, verbose=1)
print("Test loss:", score[0])
print("Test accuracy:", score[1])

Epoch 1/5
468/468 ━━━━━━━━━━━━━━━━━━━━ 1997s 4s/step - accuracy: 0.5629 - loss: 0.6893
Epoch 2/5
468/468 ━━━━━━━━━━━━━━━━━━━━ 1943s 4s/step - accuracy: 0.5843 - loss: 0.6752
Epoch 3/5
468/468 ━━━━━━━━━━━━━━━━━━━━ 1949s 4s/step - accuracy: 0.6273 - loss: 0.6500
Epoch 4/5
468/468 ━━━━━━━━━━━━━━━━━━━━ 1958s 4s/step - accuracy: 0.7212 - loss: 0.5987
Epoch 5/5
468/468 ━━━━━━━━━━━━━━━━━━━━ 2292s 5s/step - accuracy: 0.8209 - loss: 0.5171
7/7 ━━━━━━━━━━━━━━━━━━━━ 15s 2s/step - accuracy: 0.8080 - loss: 0.5138
Test loss: 0.4995110332965851
Test accuracy: 0.8299999833106995


In [6]:
model.save("CardsDetectorNew.keras")

In [12]:
import numpy as np
image = keras.utils.load_img(r'C:\Users\user\Desktop\Test3.jpg', target_size=(290, 290))
input_arr = keras.utils.img_to_array(image)
input_arr = np.array([input_arr])
prediction = model.predict(input_arr)
print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 32s 32s/step
[[0.46264392]]
